In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE_DIR = '/content/drive/MyDrive/PRISM'
RESULTS_DIR = f'{BASE_DIR}/results'

print("Setup done.")
print(f"Results dir: {RESULTS_DIR}")
print(f"Exists: {os.path.exists(RESULTS_DIR)}")

Mounted at /content/drive
Setup tamam.
Results dir: /content/drive/MyDrive/PRISM/results
Var mi: True


In [2]:
MODELS = ['CLIP', 'PLIP', 'CONCH', 'VIRCHOW2', 'UNI', 'GigaPath', 'H-Optimus-0', 'MIDNIGHT']
MODEL_KEYS = ['clip', 'plip', 'conch', 'virchow2', 'uni', 'gigapath', 'h_optimus_0', 'midnight']

DATASETS = ['PCam', 'BRACS', 'CRC', 'MHIST', 'LungHist700', 'SPIDER-Breast']
DS_KEYS  = ['pcam', 'bracs', 'crc', 'mhist', 'lunghist700', 'spider_breast']

# Naming exception (h_optimus_0_pcam yerine hoptimus_pcam kaydedilmis eski notebook'tan)
NAMING_EXCEPTIONS = {
    ('h_optimus_0', 'pcam'): 'hoptimus',
}

all_results = []
missing = []

for model, mkey in zip(MODELS, MODEL_KEYS):
    for ds, dkey in zip(DATASETS, DS_KEYS):
        actual_key = NAMING_EXCEPTIONS.get((mkey, dkey), mkey)
        res_path = f'{RESULTS_DIR}/{actual_key}_{dkey}_results.csv'

        if os.path.exists(res_path):
            df_res = pd.read_csv(res_path)
            df_res['model']   = model
            df_res['dataset'] = ds
            all_results.append(df_res)
        else:
            missing.append(f'{model} + {ds}')

df_all = pd.concat(all_results, ignore_index=True)

print(f"Loaded: {len(df_all)} satir")
print(f"Missing: {missing if missing else 'Yok'}")
print(f"\nColumns: {df_all.columns.tolist()}")
print(f"\nIlk 3 row:")
print(df_all.head(3).to_string())

Loaded: 864 satir
Missing: Yok

Column'lar: ['model', 'dataset', 'fraction', 'n_samples', 'seed', 'auroc', 'f1', 'brier', 'ece']

Ilk 3 row:
  model dataset  fraction  n_samples  seed     auroc        f1     brier       ece
0  CLIP    PCam      0.01       2621    42  0.879847  0.802245  0.147202  0.077833
1  CLIP    PCam      0.01       2621   123  0.883224  0.799496  0.147388  0.088524
2  CLIP    PCam      0.01       2621   456  0.880301  0.793774  0.147202  0.076382


In [3]:
def bootstrap_ci(values, n_bootstrap=1000, ci=95, seed=42):
    """
    Bootstrap CI: seed'lerin metriklerini N kez resample edip mean dagilimi cikar.
    Returns: (mean, lower_bound, upper_bound)
    """
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    n = len(values)
    boot_means = np.array([
        np.mean(rng.choice(values, size=n, replace=True))
        for _ in range(n_bootstrap)
    ])

    mean = np.mean(values)
    lower = np.percentile(boot_means, (100 - ci) / 2)
    upper = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return mean, lower, upper


print("Bootstrap CI hesaplaniyor (95%, n_bootstrap=1000)...")

ci_records = []
metrics = ['auroc', 'ece', 'brier', 'f1']

grouped = df_all.groupby(['model', 'dataset', 'fraction'])

for (model, dataset, fraction), group in tqdm(grouped, desc="Bootstrap CI"):
    record = {
        'model': model,
        'dataset': dataset,
        'fraction': fraction,
        'n_seeds': len(group),
    }
    for metric in metrics:
        if metric in group.columns:
            values = group[metric].values
            mean, lo, hi = bootstrap_ci(values, n_bootstrap=1000, ci=95)
            record[f'{metric}_mean'] = mean
            record[f'{metric}_ci_low'] = lo
            record[f'{metric}_ci_high'] = hi
            record[f'{metric}_ci_width'] = hi - lo
    ci_records.append(record)

ci_df = pd.DataFrame(ci_records)

CI_PATH = f'{RESULTS_DIR}/bootstrap_ci.csv'
ci_df.to_csv(CI_PATH, index=False)

print(f"\nSaved: {CI_PATH}")
print(f"Total: {len(ci_df)} (model, dataset, fraction) combination")

Bootstrap CI hesaplaniyor (95%, n_bootstrap=1000)...


Bootstrap CI: 100%|██████████| 288/288 [00:26<00:00, 10.70it/s]


Saved: /content/drive/MyDrive/PRISM/results/bootstrap_ci.csv
Total: 288 (model, dataset, fraction) kombinasyonu


In [4]:
print("=== UNI + PCam (her fraction) ===")
sample1 = ci_df[(ci_df['model']=='UNI') & (ci_df['dataset']=='PCam')]
print(sample1[['fraction', 'auroc_mean', 'auroc_ci_low', 'auroc_ci_high', 'auroc_ci_width']].to_string(index=False))

print("\n=== H-Optimus-0 + MHIST (en degisken) ===")
sample2 = ci_df[(ci_df['model']=='H-Optimus-0') & (ci_df['dataset']=='MHIST')]
print(sample2[['fraction', 'auroc_mean', 'auroc_ci_low', 'auroc_ci_high', 'auroc_ci_width']].to_string(index=False))

print("\n=== Genel ozet: ortalama CI genisligi (%1 vs %100) ===")
summary = ci_df.groupby('fraction')[
    ['auroc_ci_width', 'ece_ci_width', 'brier_ci_width']
].mean().round(4)
print(summary)

print("\n=== En genis CI'lar (en az kararli kombolar) ===")
top_uncertain = ci_df.nlargest(10, 'auroc_ci_width')[
    ['model', 'dataset', 'fraction', 'auroc_mean', 'auroc_ci_width']
]
print(top_uncertain.to_string(index=False))

=== UNI + PCam (her fraction) ===
 fraction  auroc_mean  auroc_ci_low  auroc_ci_high  auroc_ci_width
     0.01    0.981434      0.980597       0.982671    2.074261e-03
     0.05    0.983372      0.982525       0.984683    2.158124e-03
     0.10    0.983048      0.981057       0.984490    3.432565e-03
     0.25    0.981867      0.979987       0.983227    3.239997e-03
     0.50    0.981017      0.980784       0.981201    4.170202e-04
     1.00    0.978612      0.978612       0.978612    4.440892e-16

=== H-Optimus-0 + MHIST (en degisken) ===
 fraction  auroc_mean  auroc_ci_low  auroc_ci_high  auroc_ci_width
     0.01    0.659214      0.609166       0.705560        0.096394
     0.05    0.793091      0.755317       0.820327        0.065010
     0.10    0.809390      0.791928       0.827841        0.035913
     0.25    0.840943      0.822375       0.860346        0.037970
     0.50    0.867342      0.857559       0.873123        0.015564
     1.00    0.884837      0.884837       0.884837  